# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [ ]:
%pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [ ]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [ ]:
from langchain_openai import ChatOpenAI

# OpenAI() defaults to the legacy gpt-3.5-turbo-instruct completions model, and the
# gpt-3.5 family has been retired, so we use a current chat model instead
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [ ]:
doc_path =  "./datasets/state_of_the_union.txt"

In [ ]:
# encoding="utf-8": on Windows the default (cp1252) can't decode this file
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [ ]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [ ]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [ ]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [ ]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [ ]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

In [ ]:
agent.invoke("Why use ruff over flake8?")

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [ ]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [ ]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

In [ ]:
agent.invoke("Why use ruff over flake8?")

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [ ]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [ ]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.



<br>

## My dataset: Shakespeare's Sonnets

I picked `sonnets.txt` from the suggested repository - all 154 of Shakespeare's sonnets. It is a good contrast to the Ruff docs: poetry vs. technical documentation, so it should be easy to see whether the agent routes each question to the right tool.

### 1. Get the data

In [ ]:
import urllib.request

sonnets_path = "./datasets/sonnets.txt"
sonnets_url = "https://raw.githubusercontent.com/martin-gorner/tensorflow-rnn-shakespeare/master/shakespeare/sonnets.txt"

# download once, reuse the local copy afterwards
if not os.path.exists(sonnets_path):
    urllib.request.urlretrieve(sonnets_url, sonnets_path)

print(open(sonnets_path, encoding="utf-8").read()[:600])

### 2. Rebuild the vector store

Instead of fixed 1000-character chunks, we split the file on the Roman-numeral headings (`I.`, `II.`, ... `CLIV.`) so that **each chunk is exactly one sonnet**. We prefix every chunk with `Sonnet <number>` and store the number as metadata, so questions like *"What is Sonnet XVIII about?"* can match the right passage.

In [ ]:
import re
from langchain_core.documents import Document

sonnets_raw = TextLoader(sonnets_path, encoding="utf-8").load()[0].page_content

# split on lines that contain only a Roman numeral followed by a dot, e.g. "XVIII."
parts = re.split(r"\n\s*([IVXLC]+)\.\s*\n", sonnets_raw)
# parts = [intro, "I", text_1, "II", text_2, ...]
sonnet_docs = [
    Document(
        page_content=f"Sonnet {number}\n{text.strip()}",
        metadata={"source": sonnets_path, "sonnet": number},
    )
    for number, text in zip(parts[1::2], parts[2::2])
]

print(len(sonnet_docs), "sonnets")
print(sonnet_docs[17].page_content)

In [ ]:
sonnets_db = Chroma.from_documents(sonnet_docs, embeddings, collection_name="shakespeare-sonnets")

sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=sonnets_db.as_retriever(search_kwargs={"k": 4})
)

# quick sanity check of the chain on its own, before giving it to an agent
sonnets.invoke({"query": "What is the summer's day compared to in Sonnet XVIII?"})["result"]

### 3 & 4. Rewrite the tool description and rebuild the agent (agent as reasoner)

The description is what the agent reads to decide which tool to call, so it says clearly what the tool contains (poetry, Shakespeare, themes like love, beauty and time) and what it does **not** contain.

In [ ]:
sonnets_description = (
    "useful for when you need to answer questions about Shakespeare's sonnets: "
    "their text, themes (love, beauty, time, aging, death, poetry), imagery, and specific sonnets by number. "
    "It does not know anything about software or programming. "
    "Input should be a fully formed question, not referencing any obscure pronouns from the conversation before."
)
ruff_description = (
    "useful for when you need to answer questions about ruff (a python linter). "
    "Input should be a fully formed question, not referencing any obscure pronouns from the conversation before."
)

my_tools = [
    Tool(name="Shakespeare Sonnets QA System", func=sonnets.run, description=sonnets_description),
    Tool(name="Ruff QA System", func=ruff.run, description=ruff_description),
]

my_agent = initialize_agent(
    my_tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,
    handle_parsing_errors=True,  # retry instead of crashing if the LLM breaks the ReAct format
)

### 5. Put it to the test

**a) A question only the sonnets can answer**

In [ ]:
my_agent.invoke("In Shakespeare's Sonnet XVIII, why does the poet say his love is better than a summer's day?")

**b) A question only Ruff can answer**

In [ ]:
my_agent.invoke("Does ruff support Jupyter notebooks, and how do you enable it?")

**c) A multi-hop question that needs both tools**

In [ ]:
my_agent.invoke(
    "How many lines does a Shakespeare sonnet have? "
    "What is ruff's default maximum line length, and could the longest line of Sonnet XVIII fit within it?"
)

### Agent as a router (`return_direct=True`)

Same tools, but the agent now returns the tool's answer immediately instead of reasoning over it.

In [ ]:
router_tools = [
    Tool(name="Shakespeare Sonnets QA System", func=sonnets.run, description=sonnets_description, return_direct=True),
    Tool(name="Ruff QA System", func=ruff.run, description=ruff_description, return_direct=True),
]

router_agent = initialize_agent(
    router_tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,
    handle_parsing_errors=True,
)

In [ ]:
router_agent.invoke("In Shakespeare's Sonnet XVIII, why does the poet say his love is better than a summer's day?")

In [ ]:
router_agent.invoke("Does ruff support Jupyter notebooks, and how do you enable it?")

In [ ]:
# the multi-hop question again: as a router the agent can only call ONE tool and must stop there
router_agent.invoke(
    "How many lines does a Shakespeare sonnet have? "
    "What is ruff's default maximum line length, and could the longest line of Sonnet XVIII fit within it?"
)

### 6. Reflect

> ✏️ *Check these notes against your own `verbose=True` logs after running the cells above and adjust anything that differs.*

**Did the agent pick the right tool?**
- *Sonnet question* - the agent called **Shakespeare Sonnets QA System** only. Splitting the file one sonnet per chunk (with `Sonnet XVIII` in the text) made the retriever return the right poem.
- *Ruff question* - the agent called **Ruff QA System** only. The two sources are so different (poetry vs. a Python linter) that the descriptions alone are enough to route correctly.
- *Multi-hop question* - as a reasoner, the agent called **both** tools in sequence (sonnets for the line count / the text of Sonnet XVIII, Ruff for the default line length of 88) and then combined the two observations into one final answer.

**`return_direct=True` vs. not**
- Without `return_direct`, each tool call is followed by another *Thought* step: the agent can rephrase the answer, decide it needs more information, and call another tool. This costs extra LLM calls but makes multi-hop questions possible.
- With `return_direct=True`, the agent behaves as a **router**: it picks one tool, and that tool's raw answer is returned immediately. Single-source questions are faster and cheaper, but the multi-hop question only gets *half* an answer, because the agent stops after the first tool and never gets to call the second one.

**Takeaway:** the tool *description* is the routing logic. Clear, non-overlapping descriptions (including what a tool does *not* cover) are what make the agent reliable; use `return_direct=True` when every question maps to exactly one source, and leave it off when questions may need to combine sources.